# Graph Builder

Testing scripts to build a knowledge graph from entity specifications in YAML format. For Samuel Ostrove’s senior thesis in Computing and Linguistics.

In [1]:
import yaml
from abc import ABC, abstractmethod
from typing import Dict, Optional
import numpy as np
import scipy.stats as ss

## Figuring out PyYAML

In [2]:
# YAML reading
## Read single-document YAML file
test_path_1 = "./testyaml1.yml"
with open(test_path_1, 'r') as spec:
    print(yaml.safe_load(spec))

## Read multi-document YAML file
test_path_2 = "./testyaml2.yml"
with open(test_path_2, 'r') as spec:
    docs = yaml.safe_load_all(spec)
    for doc in docs:
        print(doc)

{'test1': [{'type': 'string'}, {'name': 'group'}, {'specs': [{'share': 'good'}]}]}
{'name': 'doc1'}
{'name': 'doc2'}


In [3]:
# YAML dumping
triples = [["Will", "sibling_of", "Stellan"],
           ["Hannah", "child_of", "Zoltan"]]
testdict = {'sibling_of': {'recip': 'sibling_of'}}
dump_path = "./testdump1.yml"
with open(dump_path, 'w') as path:
    yaml.dump(testdict, path)

In [ ]:
# Loading the graph into three lists
spec_path = "./testGraph1.yml"
with open(spec_path, 'r') as spec:
    docs = yaml.safe_load_all(spec)
    types, nodes, rels = docs

## Utility functions

In [2]:
## Set generation for person nodes

def randGen() -> int:
    return np.random.choice(np.arange(0, 3), size = 1, p = np.array([.3, .4, .3]))[0]

def printChecking(checking:list[tuple]):
    for i in checking:
        print(i[1].getProp('Name'), i[0].getName(), i[2].getProp('Name'))

## Infrastructure

### Nodes

In [3]:
# Base type for nodes
class node(ABC):
    """
    Basic interface for a node in the knowledge graph.
    """
    def __init__(self, name:str):
        self.properties: Dict[str, ] = dict()
        self.relations: Dict[str, set] = dict()
        self.setProp('Name', name)
        self.setProp('Generation', None)
    
    def __call__(self):
        print("This is a node, of type" + self.__class__)

    def getProp(self, prop:str):
        """
        Get the value for the specified property.
        """
        return self.properties.get(prop, 'Property not present')

    def setProp(self, prop:str, val):
        """
        Sets the specified property to the specified value.

        :param prop: Name of the property to be set
        :param val: Value to be given to the property
        """
        self.properties[prop] = val

    def getRel(self, rel:str) -> tuple[set, set]:
        """
        Get the set of nodes for which this node stands in a specified relation.
        
        :param rel: The name of a relation
        :return nodes: The set of nodes 
        :return names: The set of the names of the nodes in that relation
        """
        objects = self.relations.get(rel, set())
        if isinstance(objects, set):
            names = set(obj.getProp('Name') for obj in objects)
            return objects, names
        else:
            return objects, set()
    
    def addRel(self, rel:str, obj):
        """
        Add a relation to a node
        """
        if rel in self.relations:
            self.relations[rel].add(obj)
        else:
            self.relations[rel] = set([obj])

    def getClass(self):
        return self.__class__
    


In [4]:
# Person node class
class person(node):
    def __init__(self, name: str):
        """
        Creates a node for a person.
        
        :param name: A string, the name of the person.
        """
        super().__init__(name)
        # self.setProp('Name', name)
        self.setProp('Generation', randGen())

### Relations

In [7]:
# Relation class

class relation:
    """
    Class for relations, for internal use only.
    """
    # Hyperparameters
    defaultmax = 10

    # Method definitions
    def __init__(self, name:str):
        self.name = name
        self.subj_class = None
        self.obj_class = None
        self.recip = list()
        self.share = list()
        self.no = list()
        self.max = int()
        self.gen = int()
    
    def specify(self, subj:ABC, obj:ABC, recip:list = [], 
                share:list = [], no:list = [], max:int = defaultmax, gen:int = None):
        self.subj_class = subj
        self.obj_class = obj
        self.recip = recip
        self.share = share
        self.no = no
        self.max = max
        self.gen = gen
    
    def getName(self) -> str:
        return self.name
    
    def getClasses(self) -> tuple[ABC, ABC]:
        return self.subj_class, self.obj_class
    
    def getRecip(self) -> list:
        return self.recip
    
    def getNots(self) -> list:
        return self.no
    
    def getShare(self) -> list:
        return self.share
    
    def getMax(self) -> int:
        return self.max
    
    def getGen(self) -> int:
        return self.gen

## Building the graph

In [99]:
def tryRelation(subj:node, obj:node, rel:relation):
    """
    Interface for adding a relation between two nodes.
    """
    # Initialize the relation-checking buffer (a list of valid relations 
    # that would be put in place to satisfy creating this relation)
    checking = [(rel, subj, obj)]

    # Check that the relation is not already in place
    rel_set = subj.getRel(rel.getName())[0]
    if obj in rel_set:
        print('Relation already in place')
        return

    # Check the type requirements of the relation
    if (rel.getClasses()) != (subj.getClass(), obj.getClass()):
        print('Classes incorrect')
        return

    # Check the share requirements of the relation
    if checkShare(rel, subj, obj, checking) is False:
        print('Share check failed')
        return

    # Check the reciprocal requirements of the relation
    if checkRecip(rel, subj, obj, checking) is False:
        print('Reciprocity check failed')
        return

    # Check the not requirements of the relation
    for i in rel.getNots():
        ## Check if relation exists
        if obj in subj.getRel(i.getName())[0]:
            print('Blocking relation present')
            return
        ## Check if relation would be needed
        elif (i, subj, obj) in checking:
            print('Would require blocking relation')
            return    

    # Check the max requirement of the relation
    ## Count the number of instances for this relation and subject
    rel_count = len(rel_set)
    for i in checking:
        if (i[0] == rel) and (i[1] == subj):
            rel_count += 1
    ## Check the max requirement
    if rel_count > rel.getMax():
        print('Too many instances of relation', rel.getName())
        return
    
    # Check the generation requirement
    if rel.getGen() is not None:
        if obj.getProp('Generation') - subj.getProp('Generation') != rel.getGen():
            print('Generational mismatch')
            return

    # Add the relations from “checking”
    printChecking(checking)
    for link in checking:
        link[1].addRel(link[0].getName(), link[2])
    # addShared(rel, subj, obj)
    # for i in rel.getRecip():
    #     obj.addRel(i.getName(), subj)


### Checking functions

In [106]:
def checkRecip(rel:relation, subj:node, obj:node, checking:list) -> bool:
    """
    Checks whether a relation can be implemented as a reciprocal.

    :param rel: The original relation
    :param subj: The subject of the original relation
    :param obj: The object of the original relation
    :param checking: The buffer of assumed relations
    """
    # Iterate over the reciprocal requirements
    for recip in rel.getRecip():
        print('Checking reciprocal relation', recip.getName(), 'between nodes', obj.getProp('Name'), 'and', subj.getProp('Name'))
        # If reciprocal exists, move to next reciprocal
        if subj in obj.getRel(recip.getName())[0]:
            continue
        # If reciprocal has already been checked, move to next reciprocal
        elif (recip, obj, subj) in checking:
            continue
        else:
            # We assume that the type requirements of the reciprocal are the inverse
            # of that of the original: no need for a type check.

            # We assume that the only reciprocals of the reciprocals are the original
            # relation and its reciprocals, so no need for a reciprocal check on the
            # reciprocal.

            # Temporarily add to checking to avoid infinite loops
            checking.append((recip, obj, subj))
            # Check the share requirements of the reciprocal
            if checkShare(recip, obj, subj, checking) is False:
                print('Share check on reciprocal failed')
                return False
            checking.remove((recip, obj, subj))

            # Check the not requirements of the reciprocal
            for i in recip.getNots():
                ## Check if relation exists
                if subj in obj.getRel(i.getName())[0]:
                    print('Blocking relation present')
                    return False
                ## Check if relation would be needed
                elif (i, obj, subj) in checking:
                    print('Would require blocking relation')  
                    return False
            
            # Check the max requirement of the reciprocal
            ## Count the number of instances for the reciprocal from the object
            recip_count = len(obj.getRel(recip.getName())[0])
            for i in checking:
                if (i[0] == recip) and (i[1] == obj):
                    recip_count += 1
            ## Check the max requirement
            if recip_count > recip.getMax():
                # printChecking(checking)
                # print(obj.getRel(recip.getName())[1])
                # print(recip_count)
                print('Too many instances of reciprocal relation', recip.getName())
                return False
            
            # Check the gen requirement on the reciprocal
            if recip.getGen() is not None:
                if subj.getProp('Generation') - obj.getProp('Generation') != recip.getGen():
                #     print('Generational mismatch between', subj.getProp('Name'), 'and', obj.getProp('Name'), 'for relation', rel.getName())
                    return False
        checking.append((recip, obj, subj))
    return True

In [101]:
def checkShare(rel:relation, subj:node, obj:node, checking:list) -> bool:
    """
    Check wether the sharing requirements on a relation can be implemented

    :param rel: The original relation
    :param subj: The subject of the original relation
    :param obj: The object of the original relation
    :param checking: The buffer of assumed relations
    """
    # Iterate over shared relations
    for shared in rel.getShare():
        # print('> Shared relation being checked:', shared.getName())
        # If the shared relation is the same as the original one,
        # then check for transitivity
        if shared == rel:
            # Iterate over those who stand in the relation with
            # the original object except for the original subject,
            # and check if the relation can be implemented between
            # the original subject and that node
            for node in obj.getRel(shared.getName())[0]:
                # print('>> Name of object of shared relation:', node.getProp('Name'))
                # Get list of nodes that stand in the shared relation with the subject
                rel_set = subj.getRel(shared.getName())[0]
                # Move to next node if the node is the subject of the
                # original relation, or if the relation is already in place
                if node in rel_set:
                    continue
                elif node == subj:
                    continue
                # Check if shared relation has been checked
                elif (shared, subj, node) in checking:
                    continue
                # Check the type requirements of the relation
                elif (shared.getClasses()) != (subj.getClass(), node.getClass()):
                    print('Classes incorrect')
                    return False
                # Temporarily add to the checking list, to avoid
                # infinite loops
                checking.append((shared, subj, node))
                # Check the share requirements of the relation
                if checkShare(shared, subj, node, checking) is False:
                    print('Share check failed')
                    return False
                # Check the reciprocal requirements of the relation
                elif checkRecip(shared, subj, node, checking) is False:
                    print('Reciprocity check failed')
                    return False
                checking.remove((shared, subj, node))
                # Check the not requirements of the relation
                for block in shared.getNots():
                    ## Check if relation exists
                    if node in subj.getRel(block.getName())[0]:
                        print('Blocking relation present')
                        return False
                    ## Check if relation would be needed
                    elif (block, subj, node) in checking:
                        print('Would require blocking relation')
                        return False
                # Check the max requirement of the relation
                ## Count the number of instances for this relation and subject
                rel_count = len(rel_set)
                for i in checking:
                    if (i[0] == shared) and (i[1] == subj):
                        rel_count += 1
                ## Check the max requirement
                if rel_count > rel.getMax():
                    print('Too many instances of relation')
                    return False
                # Check the generation requirement
                if shared.getGen() is not None:
                    if node.getProp('Generation') - subj.getProp('Generation') != shared.getGen():
                        # print(print('Generational mismatch between', subj.getProp('Name'), 'and', node.getProp('Name'), 'for relation', shared.getName()),
                        #       subj.getProp('Generation'), '-', node.getProp('Generation'), 'is not', shared.getGen())
                        return False
                checking.append((shared, subj, node))
        else:
            for node in obj.getRel(shared.getName())[0]:
                # Get list of nodes that stand in the shared relation with the subject
                rel_set = subj.getRel(shared.getName())[0]
                # Move to next node if the node is the subject of the
                # original relation, or if the relation is already in place
                if node in rel_set:
                    continue
                elif node == subj:
                    continue
                # Check if shared relation has been checked
                elif (shared, subj, node) in checking:
                    continue
                # Check the type requirements of the relation
                elif (shared.getClasses()) != (subj.getClass(), node.getClass()):
                    print('Classes incorrect')
                    return False
                # Temporarily add to the checking list, to avoid
                # infinite loops
                checking.append((shared, subj, node))
                # Check the share requirements of the relation
                if checkShare(shared, subj, node, checking) is False:
                    print('Share check failed')
                    return False
                # Check the reciprocal requirements of the relation
                elif checkRecip(shared, subj, node, checking) is False:
                    print('Reciprocity check failed')
                    return False
                checking.remove((shared, subj, node))
                # Check the not requirements of the relation
                for block in shared.getNots():
                    ## Check if relation exists
                    if node in subj.getRel(block.getName())[0]:
                        print('Blocking relation present')
                        return False
                    ## Check if relation would be needed
                    elif (block, subj, node) in checking:
                        print('Would require blocking relation')
                        return False
                # Check the max requirement of the relation
                ## Count the number of instances for this relation and subject
                rel_count = len(rel_set)
                for i in checking:
                    if (i[0] == shared) and (i[1] == subj):
                        rel_count += 1
                ## Check the max requirement
                if rel_count > shared.getMax():
                    print('Too many instances of relation')
                    return False
                # Check the generation requirement
                if shared.getGen() is not None:
                    if node.getProp('Generation') - subj.getProp('Generation') != shared.getGen():
                        # print('Generational mismatch between', subj.getProp('Name'), 'and', node.getProp('Name'), 'for relation', shared.getName(),
                        #       subj.getProp('Generation'), '-', node.getProp('Generation'), 'is not', shared.getGen())
                        return False
                checking.append((shared, subj, node))
    return True

## Testing

In [102]:
sibling_of = relation('is the sibling of')
child_of = relation('is the child of')
spouse_of = relation('is the spouse of')
parent_of = relation('is the parent of')

sibling_of.specify(person, person, [sibling_of], [sibling_of, child_of], [spouse_of], 4, 0)
parent_of.specify(person, person, [child_of], max = 5, gen = 1)
child_of.specify(person, person, [parent_of], max = 2, gen = -1)
spouse_of.specify(person, person, [spouse_of], [parent_of], [sibling_of, parent_of, child_of], 1, 0)

In [93]:
John = person('John')
Jane = person('Jane')
John.addRel('child', Jane)

for i in John.getRel('child')[0]:
    print(i)

In [107]:
Shulamit = person('Shulamit')
Shulamit.setProp('Generation', 2)
Ariela = person('Ariela')
Ariela.setProp('Generation', 1)
Samuel = person('Samuel')
Samuel.setProp('Generation', 2)
Michael = person('Michael')
Michael.setProp('Generation', 1)

In [ ]:
tryRelation(Shulamit, Ariela, child_of)
tryRelation(Samuel, Shulamit, sibling_of)
tryRelation(Michael, Ariela, spouse_of)

Checking reciprocal relation is the parent of between nodes Ariela and Shulamit
Shulamit is the child of Ariela
Ariela is the parent of Shulamit
{'Ariela'}
{'Shulamit'}
